# Hourly Rainfall Forecasting In EAT

This notebook trains and evaluates the dedicated hourly rainfall model from `historical_weather_data_hourly.csv` in the `weather_irrigation_hourly` folder. All timestamps are normalized to EAT (`Africa/Nairobi`) before feature engineering. Historical weather columns are stored for auditing, but the deployed model only uses date, month, and time at inference.

## 1. Imports

In [ ]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression, TweedieRegressor
from sklearn.metrics import accuracy_score, brier_score_loss, mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler

BASE_DIR = Path.cwd()
DATA_PATH = BASE_DIR / 'historical_weather_data_hourly.csv'
MODEL_PATH = BASE_DIR / 'hourly_rainfall_forecaster.joblib'
METRICS_PATH = BASE_DIR / 'hourly_rainfall_metrics.json'
RAIN_THRESHOLD_MM = 0.1
np.random.seed(42)


## 2. Load EAT-Normalized Hourly History

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        'historical_weather_data_hourly.csv was not found. Run fetch_hourly_weather_data.py first.'
    )

df = pd.read_csv(DATA_PATH, parse_dates=['time'])
df = df.sort_values('time').reset_index(drop=True)
df['rain_mm'] = df['rain_mm'].clip(lower=0)
df['rain_occurrence'] = (df['rain_mm'] >= RAIN_THRESHOLD_MM).astype(int)
display_columns = [
    'time',
    'time_utc',
    'precipitation_probability',
    'rain_mm',
    'cloud_cover_total_percent',
    'evapotranspiration_mm',
    'soil_temperature_c',
    'soil_moisture_m3m3',
]
print(df[display_columns].head())
print(f'Loaded {len(df):,} EAT hourly rows from {df.time.min()} to {df.time.max()}')


## 3. Calendar Feature Engineering

In [ ]:
def cyclical_columns(values: pd.Series, period: float, prefix: str, harmonics: int = 1) -> dict[str, np.ndarray]:
    angles = 2 * np.pi * values.to_numpy(dtype=float) / period
    features = {}
    for harmonic in range(1, harmonics + 1):
        features[f'{prefix}_sin_{harmonic}'] = np.sin(harmonic * angles)
        features[f'{prefix}_cos_{harmonic}'] = np.cos(harmonic * angles)
    return features

def build_climatology(frame: pd.DataFrame) -> dict:
    train = frame.copy()
    month_hour = train.groupby(['month_eat', 'hour_eat'])[['rain_mm', 'rain_occurrence']].mean().reset_index()
    doy_hour = train.groupby(['day_of_year_eat', 'hour_eat'])[['rain_mm', 'rain_occurrence']].mean().reset_index()
    return {
        'month_hour_amount': {(int(r.month_eat), int(r.hour_eat)): float(r.rain_mm) for r in month_hour.itertuples()},
        'month_hour_probability': {(int(r.month_eat), int(r.hour_eat)): float(r.rain_occurrence) for r in month_hour.itertuples()},
        'doy_hour_amount': {(int(r.day_of_year_eat), int(r.hour_eat)): float(r.rain_mm) for r in doy_hour.itertuples()},
        'doy_hour_probability': {(int(r.day_of_year_eat), int(r.hour_eat)): float(r.rain_occurrence) for r in doy_hour.itertuples()},
        'global_amount': float(train['rain_mm'].mean()),
        'global_probability': float(train['rain_occurrence'].mean()),
    }

def build_features(timestamps: pd.Series, climatology: dict) -> pd.DataFrame:
    ts = pd.to_datetime(timestamps)
    month = ts.dt.month
    hour = ts.dt.hour
    minute = ts.dt.minute
    day_of_year = ts.dt.dayofyear
    day_of_week = ts.dt.dayofweek
    day_of_month = ts.dt.day
    year_index = ts.dt.year - ts.dt.year.min()
    month_hour_keys = list(zip(month, hour))
    doy_hour_keys = list(zip(day_of_year, hour))

    features = {
        'year_index': year_index.to_numpy(dtype=float),
        'is_weekend': (day_of_week >= 5).astype(float).to_numpy(),
        'month_hour_amount_climatology': np.array([climatology['month_hour_amount'].get(k, climatology['global_amount']) for k in month_hour_keys]),
        'month_hour_probability_climatology': np.array([climatology['month_hour_probability'].get(k, climatology['global_probability']) for k in month_hour_keys]),
        'doy_hour_amount_climatology': np.array([climatology['doy_hour_amount'].get(k, climatology['global_amount']) for k in doy_hour_keys]),
        'doy_hour_probability_climatology': np.array([climatology['doy_hour_probability'].get(k, climatology['global_probability']) for k in doy_hour_keys]),
    }
    features.update(cyclical_columns(month, 12.0, 'month'))
    features.update(cyclical_columns(hour + minute / 60.0, 24.0, 'hour', harmonics=2))
    features.update(cyclical_columns(day_of_year, 365.25, 'doy', harmonics=2))
    features.update(cyclical_columns(day_of_week, 7.0, 'dow'))
    features.update(cyclical_columns(day_of_month, 31.0, 'dom'))
    return pd.DataFrame(features, index=ts.index).astype(float)


## 4. Time-Based Split

In [ ]:
split_idx = int(len(df) * 0.8)
train_df = df.iloc[:split_idx].copy()
test_df = df.iloc[split_idx:].copy()
climatology = build_climatology(train_df)

X_train = build_features(train_df['time'], climatology)
X_test = build_features(test_df['time'], climatology)
y_train_amount = train_df['rain_mm'].to_numpy(dtype=float)
y_test_amount = test_df['rain_mm'].to_numpy(dtype=float)
y_train_rain = train_df['rain_occurrence'].to_numpy(dtype=int)
y_test_rain = test_df['rain_occurrence'].to_numpy(dtype=int)

month_hour_keys_test = list(zip(test_df['month_eat'], test_df['hour_eat']))
baseline_amount = np.array([climatology['month_hour_amount'].get(k, climatology['global_amount']) for k in month_hour_keys_test])
baseline_probability = np.array([climatology['month_hour_probability'].get(k, climatology['global_probability']) for k in month_hour_keys_test])

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


## 5. Hyperparameter Search

In [ ]:
def amount_metrics(y_true, y_pred):
    return {
        'mae_mm': float(mean_absolute_error(y_true, y_pred)),
        'rmse_mm': float(root_mean_squared_error(y_true, y_pred)),
        'r2': float(r2_score(y_true, y_pred)),
    }

def probability_metrics(y_true, y_prob):
    return {
        'brier_score': float(brier_score_loss(y_true, y_prob)),
        'accuracy': float(accuracy_score(y_true, y_prob >= 0.5)),
    }

def search_tweedie(X, y):
    candidates = [{'alpha': alpha, 'power': 1.5} for alpha in (0.01, 0.1, 0.5, 1.0, 2.0)]
    splitter = TimeSeriesSplit(n_splits=5)
    best_params = candidates[0]
    best_score = float('inf')
    for params in candidates:
        scores = []
        for train_idx, valid_idx in splitter.split(X):
            model = TweedieRegressor(link='log', max_iter=1000, tol=1e-5, **params)
            model.fit(X.iloc[train_idx], y[train_idx])
            pred = model.predict(X.iloc[valid_idx]).clip(min=0)
            scores.append(mean_absolute_error(y[valid_idx], pred))
        score = float(np.mean(scores))
        if score < best_score:
            best_score = score
            best_params = params
    return best_params

def search_logistic(X, y):
    candidates = [{'C': c} for c in (0.1, 0.5, 1.0, 2.0, 5.0)]
    splitter = TimeSeriesSplit(n_splits=5)
    best_params = candidates[0]
    best_score = float('inf')
    for params in candidates:
        scores = []
        for train_idx, valid_idx in splitter.split(X):
            model = LogisticRegression(max_iter=2000, solver='lbfgs', random_state=42, **params)
            model.fit(X.iloc[train_idx], y[train_idx])
            pred_prob = model.predict_proba(X.iloc[valid_idx])[:, 1]
            scores.append(brier_score_loss(y[valid_idx], pred_prob))
        score = float(np.mean(scores))
        if score < best_score:
            best_score = score
            best_params = params
    return best_params

X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X_train.columns)
best_regressor_params = search_tweedie(X_train_scaled_df, y_train_amount)
best_classifier_params = search_logistic(X_train_scaled_df, y_train_rain)


## 6. Train And Evaluate

In [ ]:
amount_model = TweedieRegressor(link='log', max_iter=1000, tol=1e-5, **best_regressor_params)
probability_model = LogisticRegression(max_iter=2000, solver='lbfgs', random_state=42, **best_classifier_params)

amount_model.fit(X_train_scaled, y_train_amount)
probability_model.fit(X_train_scaled, y_train_rain)

amount_pred = amount_model.predict(X_test_scaled).clip(min=0)
probability_pred = probability_model.predict_proba(X_test_scaled)[:, 1]

metrics = {
    'train_rows': int(len(train_df)),
    'test_rows': int(len(test_df)),
    'best_regressor_params': best_regressor_params,
    'best_classifier_params': best_classifier_params,
    'ml_amount': amount_metrics(y_test_amount, amount_pred),
    'baseline_amount': amount_metrics(y_test_amount, baseline_amount),
    'ml_probability': probability_metrics(y_test_rain, probability_pred),
    'baseline_probability': probability_metrics(y_test_rain, baseline_probability),
}
metrics['selected_amount_model'] = (
    'ml_amount_model'
    if metrics['ml_amount']['mae_mm'] < metrics['baseline_amount']['mae_mm']
    and metrics['ml_amount']['rmse_mm'] < metrics['baseline_amount']['rmse_mm']
    else 'month_hour_climatology'
)
metrics['selected_probability_model'] = (
    'ml_probability_model'
    if metrics['ml_probability']['brier_score'] < metrics['baseline_probability']['brier_score']
    else 'month_hour_climatology'
)
print(json.dumps(metrics, indent=2))


## 7. Export Artifact

In [ ]:
artifact = {
    'amount_model': amount_model,
    'probability_model': probability_model,
    'scaler': scaler,
    'feature_columns': list(X_train.columns),
    'climatology': climatology,
    'selected_amount_model': metrics['selected_amount_model'],
    'selected_probability_model': metrics['selected_probability_model'],
    'rain_threshold_mm': RAIN_THRESHOLD_MM,
    'evaluation': metrics,
}

joblib.dump(artifact, MODEL_PATH)
METRICS_PATH.write_text(json.dumps(metrics, indent=2))
print(f'Saved {MODEL_PATH.name}')
print(f'Saved {METRICS_PATH.name}')
